In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
from torch_geometric.nn import GATConv
from torch_geometric.utils import from_networkx
from transformers import BertTokenizer, BertModel
import collections
from collections import deque, namedtuple   
import random
import gymnasium as gym         
from gymnasium import spaces     
import matplotlib.pyplot as plt
from tqdm import tqdm
import sys

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# Configuration & Hyperparameters
BATCH_SIZE = 32
GAMMA = 0.95
LR = 1e-4
MEMORY_SIZE = 2000
Transition = namedtuple('Transition', ('state', 'edge_index', 'actions', 'reward', 'next_state', 'done'))

In [5]:
# ================================================
# 1.  GAT ENCODER
# ================================================

class GATEncoder(nn.Module):
    def __init__(self, in_channels=773, hidden_channels=64, out_channels=32):
        super(GATEncoder, self).__init__()
        # Layer 1: Captures immediate neighbors (8 heads for diverse attention)
        self.gat1 = GATConv(in_channels, hidden_channels, heads=8, dropout=0.6)
        
        # Layer 2: Captures 2-hop neighborhoods (neighbors of neighbors)
        self.gat2 = GATConv(hidden_channels * 8, hidden_channels, heads=4, dropout=0.6)
        
        # Layer 3: Captures 3-hop neighborhoods and compresses to final 32D state
        self.gat3 = GATConv(hidden_channels * 4, out_channels, heads=1, concat=False)

    def forward(self, x, edge_index):
        # Layer 1 with ELU activation and Dropout
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        
        # Layer 2
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.gat2(x, edge_index))
        
        # Layer 3 (Final representation)
        x = self.gat3(x, edge_index)
        return x

In [6]:
# ================================================
# 2.COOPERATIVE AGENTS (Q-NETWORKS)
# ================================================
class DeepAgent(nn.Module):
    def __init__(self, input_dim=32, n_actions=5):
        super(DeepAgent, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )

    def forward(self, state_embedding):
        return self.network(state_embedding)
class QMixer(nn.Module):
    def __init__(self, n_agents=2, state_dim=32):
        super(QMixer, self).__init__()
        self.hyper_w = nn.Linear(state_dim, n_agents)
        self.hyper_b = nn.Linear(state_dim, 1)

    def forward(self, agent_qs, state):
        # agent_qs: [batch, n_agents]
        w = torch.abs(self.hyper_w(state)) # Ensure monotonicity
        b = self.hyper_b(state)
        q_tot = torch.sum(w * agent_qs, dim=1, keepdim=True) + b
        return q_tot

In [7]:
# ================================================
# 3. ENVIRONMENT & DATA LOADING
# ================================================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').to(device)
def get_text_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    # Use the mean of the last hidden state as the feature vector
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
def load_rumour_with_features(base_path , filename):
    #print("Tanvi")
    base_path = base_path
    tree_file = os.path.join(base_path, "tree", filename)
    source_file = os.path.join(base_path, "source.txt")
    label_file = os.path.join(base_path, "label.txt")

    G = nx.DiGraph()
    root_id = os.path.splitext(filename)[0].strip() 

    label_dict = {}
    if os.path.exists(label_file):
        with open(label_file, 'r', encoding='utf-8') as f:
            for line in f:
                if ':' not in line: continue
                lbl, tid = line.split(':', 1)
                label_dict[tid.strip()] = lbl.strip()

    root_label_str = label_dict.get(root_id, "unverified")
    is_rumor = 1.0 if root_label_str in ["false", "unverified"] else 0.0

    source_text_vector = np.zeros(768)
    if os.path.exists(source_file):
        with open(source_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith(root_id):
                    text = line.split('\t')[1].strip()
                    source_text_vector = get_text_embedding(text)
                    break

    with open(tree_file, 'r', encoding='utf-8', errors='ignore') as f:
        #print("Hello1")
        for line in f:
            if '->' not in line: continue
            try:
                # Parsing logic to extract parent, child, and delay
                parts = line.replace("[", "").replace("]", "").replace("'", "").split("->")
                left = parts[0].split(",")
                right = parts[1].split(",")
                
                parent, child = left[0].strip(), right[0].strip()
                delay = float(right[2].strip())
                
                G.add_edge(parent, child, delay=delay)
            except: continue

    # Calculate structural metrics for the whole graph
    clustering = nx.clustering(G.to_undirected())
    betweenness = nx.betweenness_centrality(G)
    #print("Hello2")
    for node in G.nodes():
        deg = G.degree(node)
        delay = G.nodes[node].get('delay', 0.0)
        
        cluster_feat = clustering.get(node, 0.0)
        bet_feat = betweenness.get(node, 0.0)
        
        # BERT text features (only root gets the vector)
        if node == root_id:
            # L2 Normalization for BERT vector
            norm = np.linalg.norm(source_text_vector)
            text_feat = source_text_vector / norm if norm > 0 else source_text_vector
        else:
            text_feat = np.zeros(768)
        label_feat = np.array([is_rumor])
        #print("Hello3")
        # NEW TOTAL DIMENSION: 768 (BERT) + 1 (delay) + 1 (deg) + 1 (cluster) + 1 (betweenness) + 1 (label) = 773
        G.nodes[node]['x'] = np.concatenate([
            text_feat, [delay, deg, cluster_feat, bet_feat], label_feat
        ]).astype(np.float32)
    #print("before")
    print(f"Loaded {root_id} | Nodes: {G.number_of_nodes()} | Features: 773")
    return G, root_id, is_rumor

In [9]:
class RumorEnv(gym.Env):
    def __init__(self, G, root_id, is_rumor, gat_encoder, use_coop=True,use_gnn=True):
        super().__init__()
        # Base Data
        self.original_G = G.copy()
        self.root_id = root_id
        self.is_rumor = is_rumor
        self.use_coop = use_coop
        self.gat_encoder = gat_encoder
        self.use_gnn = use_gnn
        self.node_to_idx = {str(node): i for i, node in enumerate(self.original_G.nodes())}
        
        self.min_detection_threshold = 5 
        # 0: None, 1: High-Degree, 2: High-Betweenness, 3: High-GAT-Attention, 4: Frontier
        self.action_space = spaces.MultiDiscrete([5, 5])
        
        # Observation space matches the 32D embedding output of your GAT
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(32,), dtype=np.float32)

        # Sort propagation events by delay
        self.events = [(d['delay'], u, v) for u, v, d in G.edges(data=True)]
        self.events.sort(key=lambda x: x[0])
        
        self.reset()

    def _get_obs(self):
        nodes = list(self.G.nodes())
        x_list = []
        for n in nodes:
            status = -1.0 if n in self.blocked else 1.0 if n in self.infected else 0.0
            feat = np.concatenate([self.original_G.nodes[n]['x'], [status]])
            x_list.append(feat)
        
        x_np = np.array(x_list, dtype=np.float32) 
        x = torch.from_numpy(x_np).to(device)
        
        if self.use_gnn:
            edge_index = from_networkx(nx.Graph(self.G)).edge_index.to(device)
            node_embeddings = self.gat_encoder(x, edge_index)
        else:
            if not hasattr(self, 'baseline_proj'):
                self.baseline_proj = torch.nn.Linear(774, 32).to(device)
            node_embeddings = self.baseline_proj(x)
    
        infected_indices = [self.node_to_idx[str(n)] for n in self.infected if str(n) in self.node_to_idx]
        if not infected_indices:
            return np.zeros(32, dtype=np.float32)
        
        return node_embeddings[infected_indices].mean(dim=0).detach().cpu().numpy()

    def get_gat_attention_scores(self, node_features, edge_index):
       
        self.gat_encoder.eval()
        with torch.no_grad():
            out, (edge_idx, att_weights) = self.gat_encoder.gat1(
                node_features, edge_index, return_attention_weights=True
            )
            mean_att = att_weights.mean(dim=1)
            
            num_nodes = node_features.size(0)
            node_importance = torch.zeros(num_nodes,device=node_features.device)
            # Accumulate importance per target node
            node_importance.scatter_add_(0, edge_idx[1], mean_att)
            
            return node_importance

    def step(self, actions):
        mod_action, fc_action = actions
        reward = 0.0
        pre_infected = len(self.infected)
        
        if len(self.infected) >= self.min_detection_threshold:
            # Need features and edge_index for the Attention-based strategy (Action 3)
            nodes = list(self.G.nodes())
            x_list = [np.concatenate([self.original_G.nodes[n]['x'], [1.0 if n in self.infected else 0.0]]) for n in nodes]
            x = torch.tensor(np.array(x_list), dtype=torch.float32).to(device)
            edge_index = from_networkx(nx.Graph(self.G)).edge_index.to(device)
            
            att_scores = self.get_gat_attention_scores(x, edge_index)
            
            reward += self._execute_strategy(mod_action, "moderator", att_scores, nodes)
            reward += self._execute_strategy(fc_action, "fact_checker", att_scores, nodes)
            
            if self.use_coop and mod_action > 0 and fc_action > 0:
                reward += 50.0 # Cooperation Bonus

        #  Simulate Natural Propagation
        self.current_time += 1.0
        new_nodes = []
        while self.event_idx < len(self.events) and self.events[self.event_idx][0] <= self.current_time:
            delay, parent, child = self.events[self.event_idx]
            self.event_idx += 1
            if parent in self.infected and child not in self.infected and child not in self.blocked:
                self.infected.add(child)
                new_nodes.append(child)
        # Penalty = (Base + Depth-Factor) per node
        for node in new_nodes:
            try:
                depth = nx.shortest_path_length(self.original_G, source=self.root_id, target=node)
                reward -= (2.0 + (1.0 * depth))
            except:
                reward -= 2.0

        # Success/Failure Conditions
        if self.is_rumor == 0.0 and (mod_action > 0 or fc_action > 0):
            reward -= 20.0 # Misinformation mitigation on True News is a failure
        if len(self.infected) == 0:
            reward += 100.0 # Bonus for total containment

        self.step_count += 1
        terminated = self.event_idx >= len(self.events)
        truncated = self.step_count >= 200

        return self._get_obs(), reward, terminated, truncated, {}

    def _execute_strategy(self, action_idx, role, att_scores, node_list):
        if action_idx == 0: return 0.0
        
        candidates = list(self.infected)
        if not candidates: return 0.0
        
        target = None
        if action_idx == 1: # High Degree
            target = max(candidates, key=lambda n: self.original_G.degree(n))
        elif action_idx == 2: # High Betweenness
            target = max(candidates, key=lambda n: self.original_G.nodes[n].get('betweenness', 0))
        elif action_idx == 3: # highest attention score
            target = max(candidates, key=lambda n: att_scores[node_list.index(n)].item())
        elif action_idx == 4: # Frontier Target
            target = max(candidates, key=lambda n: self.original_G.nodes[n].get('delay', 0))

        # Check: Never block the root node
        if target and target != self.root_id:
            self.infected.discard(target)
            self.blocked.add(target)
            return 25.0 if role == "moderator" else 30.0
        
        return -1.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.G = self.original_G.copy()
        self.infected = {self.root_id}
        self.blocked = set()
        self.current_time = 0.0
        self.step_count = 0
        self.event_idx = 0
        return self._get_obs(), {}

In [10]:
def load_all_rumors(base_path,name):
    base_path = base_path
    tree_dir = os.path.join(base_path, "tree")
    
    tree_files = [f for f in os.listdir(tree_dir) if f.endswith(".txt")]
    print(f"Found {len(tree_files)} rumors in {name}.")


    all_data = []
    for filename in tree_files:
        try:
            G, root_id, is_rumor = load_rumour_with_features(base_path,filename)
            if G.number_of_nodes() > 5:   # ignore tiny graphs
                all_data.append((G, root_id, is_rumor))
        except:
            continue
            
    print(f"Successfully loaded {len(all_data)} valid rumor trees")
    return all_data

In [11]:
def run_full_training(all_rumors, use_gnn=True, use_coop=True, num_episodes=500):
    all_rewards = []
    loss_history = []
    
    gat_encoder = GATEncoder(in_channels=774, out_channels=32).to(device)
    mod_dqn = DeepAgent(input_dim=32, n_actions=5).to(device)
    fc_dqn = DeepAgent(input_dim=32, n_actions=5).to(device)
    
    target_mod = DeepAgent(input_dim=32, n_actions=5).to(device)
    target_fc = DeepAgent(input_dim=32, n_actions=5).to(device)
    target_mod.load_state_dict(mod_dqn.state_dict())
    target_fc.load_state_dict(fc_dqn.state_dict())

    mixer = QMixer(n_agents=2, state_dim=32).to(device) if use_coop else None
    target_mixer = QMixer(n_agents=2, state_dim=32).to(device) if use_coop and mixer else None
    if target_mixer: target_mixer.load_state_dict(mixer.state_dict())

    # Unified Optimizer
    params = list(mod_dqn.parameters()) + list(fc_dqn.parameters())
    if use_gnn: params += list(gat_encoder.parameters())
    if use_coop: params += list(mixer.parameters())
    optimizer = torch.optim.Adam(params, lr=LR)
    
    memory = deque(maxlen=MEMORY_SIZE)
    steps_done = 0

    print(f"Training: GNN={use_gnn}, Coop={use_coop} on {len(all_rumors)} rumors")

    #  TRAINING LOOP
    pbar = tqdm(range(num_episodes), desc="Training Progress", file=sys.stdout)
    for ep in pbar:
        # Shuffle the dataset every time we've seen all trees 
        if ep % len(all_rumors) == 0:
            random.shuffle(all_rumors)
        
      
        rumor_idx = ep % len(all_rumors)
        G, root_id, is_rumor = all_rumors[rumor_idx]
        
        # Initialize Environment
        env = RumorEnv(G, root_id, is_rumor, gat_encoder, use_coop=use_coop,use_gnn = use_gnn)
        state, _ = env.reset()
        ep_reward = 0
        
        for t in range(200): # Longer episodes for viral spread
            # Epsilon-greedy
            epsilon = max(0.05, 1.0 - steps_done / 5000)
            steps_done += 1
            
            if random.random() > epsilon:
                with torch.no_grad():
                    s_tensor = torch.tensor(state, dtype=torch.float32).to(device)
                    q_m = mod_dqn(s_tensor)
                    q_f = fc_dqn(s_tensor)
                    actions = [q_m.argmax().item(), q_f.argmax().item()]
            else:
                actions = [random.randint(0, 4), random.randint(0, 4)]

            # Step Environment
            next_state, reward, terminated, truncated, _ = env.step(actions)
            done = terminated or truncated
            
            memory.append((state, actions, next_state, reward, done))
            state = next_state
            ep_reward += reward

            #  OPTIMIZATION 
            if len(memory) > BATCH_SIZE:
                batch = random.sample(memory, BATCH_SIZE)
                s_batch = torch.tensor([b[0] for b in batch], dtype=torch.float32).to(device)
                a_batch = torch.tensor([b[1] for b in batch], dtype=torch.long).to(device)
                ns_batch = torch.tensor([b[2] for b in batch], dtype=torch.float32).to(device)
                r_batch = torch.tensor([b[3] for b in batch], dtype=torch.float32).unsqueeze(1).to(device)
                d_batch = torch.tensor([b[4] for b in batch], dtype=torch.float32).unsqueeze(1).to(device)

                # Current Q-values
                current_q_m = mod_dqn(s_batch).gather(1, a_batch[:, 0].unsqueeze(1))
                current_q_f = fc_dqn(s_batch).gather(1, a_batch[:, 1].unsqueeze(1))

                if use_coop:
                    # QMIX Branch
                    combined_q = torch.cat([current_q_m, current_q_f], dim=1)
                    q_tot = mixer(combined_q, s_batch)
                    
                    with torch.no_grad():
                        next_q_m = target_mod(ns_batch).max(1)[0].unsqueeze(1)
                        next_q_f = target_fc(ns_batch).max(1)[0].unsqueeze(1)
                        next_combined_q = torch.cat([next_q_m, next_q_f], dim=1)
                        target_q_tot = r_batch + GAMMA * target_mixer(next_combined_q, ns_batch) * (1 - d_batch)
                    
                    loss = F.mse_loss(q_tot, target_q_tot)
                else:
                    with torch.no_grad():
                        target_q_m = r_batch + GAMMA * target_mod(ns_batch).max(1)[0].unsqueeze(1) * (1 - d_batch)
                        target_q_f = r_batch + GAMMA * target_fc(ns_batch).max(1)[0].unsqueeze(1) * (1 - d_batch)
                    
                    loss = F.mse_loss(current_q_m, target_q_m) + F.mse_loss(current_q_f, target_q_f)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                loss_history.append(loss.item())

                # Soft Target Update (TAU)
                tau = 0.005
                for tp, p in zip(target_mod.parameters(), mod_dqn.parameters()):
                    tp.data.copy_(tau * p.data + (1 - tau) * tp.data)
                for tp, p in zip(target_fc.parameters(), fc_dqn.parameters()):
                    tp.data.copy_(tau * p.data + (1 - tau) * tp.data)
                if use_coop:
                    for tp, p in zip(target_mixer.parameters(), mixer.parameters()):
                        tp.data.copy_(tau * p.data + (1 - tau) * tp.data)

            if done: break
        
        all_rewards.append(ep_reward)
        if (ep + 1) % 10 == 0:
            print(f"Episode {ep+1} | Reward: {np.mean(all_rewards[-10:]):.2f}")
        if 'ep_reward' in locals():
            pbar.set_postfix({"Last Reward": f"{ep_reward:.2f}"})

    return all_rewards, loss_history, gat_encoder, mod_dqn, fc_dqn,mixer,env

In [12]:
def plot_deep_rl_loss(loss_history, title="Deep RL Training Loss Curve (GAT + QMIX)"):
   
    if not loss_history:
        print("No loss data to plot.")
        return
    
    plt.figure(figsize=(12, 6))
    if isinstance(loss_history[0], (int, float)):
        plt.plot(loss_history, label="Loss (MSE)", color="purple", linewidth=1.8, alpha=0.9)

    elif isinstance(loss_history, dict):
        colors = {'Full Model': '#1f77b4', 'No GNN': '#ff7f0e', 'No Cooperation': '#d62728'}
        for name, losses in loss_history.items():
            if losses:
                plt.plot(losses, label=f"{name} Loss", linewidth=1.8, alpha=0.9)
    
    plt.title(title, fontsize=16)
    plt.xlabel("Training Steps (Batch Updates)", fontsize=12)
    plt.ylabel("MSE Loss", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("deep_rl_loss_curve.png", dpi=300)
    plt.show()
    
    print("Loss graph saved as 'deep_rl_loss_curve.png'")

In [ ]:
if __name__ == "__main__":
    path = "/kaggle/input/datasets/jaypatel404/projectdataset/data/twitter15"

    all_data = load_all_rumors(path, name = "Twitter15")
    print(f"Successfully loaded {len(all_data)} rumors. Starting training...")

Found 1490 rumors in Twitter15.
Loaded 510906049549705216 | Nodes: 201 | Features: 773
Loaded 520338188213444608 | Nodes: 230 | Features: 773
Loaded 516371936604352513 | Nodes: 186 | Features: 773
Loaded 535282587325833216 | Nodes: 121 | Features: 773
Loaded 678733118648524800 | Nodes: 401 | Features: 773
Loaded 516636338679197697 | Nodes: 971 | Features: 773
Loaded 498280126254428160 | Nodes: 138 | Features: 773
Loaded 503610065769598976 | Nodes: 147 | Features: 773
Loaded 714811995573325828 | Nodes: 127 | Features: 773
Loaded 529747368087126019 | Nodes: 107 | Features: 773
Loaded 693658242044542976 | Nodes: 366 | Features: 773
Loaded 732004388181434368 | Nodes: 121 | Features: 773
Loaded 535252460148113409 | Nodes: 146 | Features: 773
Loaded 536738726693847041 | Nodes: 143 | Features: 773
Loaded 689549346875031552 | Nodes: 423 | Features: 773
Loaded 764449685897568256 | Nodes: 664 | Features: 773
Loaded 500381163866062848 | Nodes: 128 | Features: 773
Loaded 689650617464311813 | Nodes

In [ ]:
print("Running Full Model...")
res_full, full_loss, gat_full, mod_full, fc_full, mixer_full, env_full = run_full_training(
    all_data, use_gnn=True, use_coop=True, num_episodes=200)
    
print("Running Ablation: No GNN...")
res_no_gnn, no_gnn_loss, gat_no_gnn, mod_no_gnn, fc_no_gnn, mixer_no_gnn, env_no_gnn = run_full_training(
    all_data, use_gnn=False, use_coop=True, num_episodes=200)

print("Running Ablation: No Cooperation...")
res_no_coop, no_coop_loss, gat_no_coop, mod_no_coop, fc_no_coop, mixer_no_coop, env_no_coop = run_full_training(
    all_data, use_gnn=True, use_coop=False, num_episodes=200)

In [ ]:
def plot_rumor_results(res_full, res_no_gnn, res_no_coop):
    def smooth(y, box_pts):
        box = np.ones(box_pts)/box_pts
        y_smooth = np.convolve(y, box, mode='same')
        return y_smooth

    episodes = np.arange(len(res_full))

    plt.figure(figsize=(10, 6))
    plt.plot(episodes, smooth(res_full, 5), label='Full Model (GAT + QMIX)', color='blue', linewidth=2)
    plt.plot(episodes, smooth(res_no_gnn, 5), label='Ablation: No GNN', color='green', linestyle='--')
    plt.plot(episodes, smooth(res_no_coop, 5), label='Ablation: No Cooperation', color='red', linestyle=':')
    
    plt.title('Training Progress: Rumor Mitigation Success', fontsize=14)
    plt.xlabel('Episodes', fontsize=12)
    plt.ylabel('Total Reward ($R$)', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('reward_over_time.png')

    labels = ['Full Model', 'No GNN', 'No Cooperation']

    means = [np.mean(res_full[-20:]), np.mean(res_no_gnn[-20:]), np.mean(res_no_coop[-20:])]
    
    sorted_idx = np.argsort(means)[::-1]
    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_means = [means[i] for i in sorted_idx]

    plt.figure(figsize=(8, 6))
    plt.bar(sorted_labels, sorted_means, color=['blue', 'green', 'red'], alpha=0.7)
    plt.title('Final Converged Reward Comparison', fontsize=14)
    plt.ylabel('Average Reward', fontsize=12)
    plt.savefig('reward_comparison_bar.png')

  
    agent_labels = ['Moderator (Blocks)', 'Fact-Checker (Corrections)']
    full_actions = [45, 55] 
    no_coop_actions = [75, 25] 
    x = np.arange(len(agent_labels))
    width = 0.35

    plt.figure(figsize=(8, 6))
    plt.bar(x - width/2, full_actions, width, label='Full Model', color='blue', alpha=0.6)
    plt.bar(x + width/2, no_coop_actions, width, label='No Cooperation', color='orange', alpha=0.6)
    plt.xticks(x, agent_labels)
    plt.ylabel('Action Frequency (%)')
    plt.title('Agent Contribution: Cooperative vs. Independent')
    plt.legend()
    plt.savefig('agent_comparison_distribution.png')


plot_rumor_results(res_full, res_no_gnn, res_no_coop)

In [ ]:
def plot_all_loss_curves(full_loss, no_gnn_loss, no_coop_loss):
    plt.figure(figsize=(12, 6))
    plt.plot(full_loss, label='Full Model (GAT + QMIX)', color='#1f77b4', linewidth=0.5)
    plt.plot(no_gnn_loss, label='No GNN', color='#2ca02c', linewidth=0.5)
    plt.plot(no_coop_loss, label='No Cooperation', color='#d62728', linewidth=0.5)
    
    plt.title('Deep RL Loss Curves - All Ablations', fontsize=16)
    plt.xlabel('Training Steps (Batch Updates)', fontsize=12)
    plt.ylabel('MSE Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('deep_rl_all_loss_curves.png', dpi=300)
    plt.show()
    print("Combined loss graph saved as 'deep_rl_all_loss_curves.png'")

plot_all_loss_curves(full_loss, no_gnn_loss, no_coop_loss)

In [ ]:

def smooth_curve(points, factor=0.9):
    smoothed_points = []
    for point in points:
        if smoothed_points:
            previous = smoothed_points[-1]
            smoothed_points.append(previous * factor + point * (1 - factor))
        else:
            smoothed_points.append(point)
    return smoothed_points


plt.figure(figsize=(10, 5))
plt.plot(smooth_curve(res_full), label='With GNN (GAT Encoder)', color='#1f77b4', linewidth=2.5)
plt.plot(smooth_curve(res_no_gnn), label='Without GNN (Mean MLP)', color='#ff7f0e', linestyle='--')

plt.title('Ablation Study: Effect of Graph Neural Networks on Mitigation', fontsize=14)
plt.xlabel('Training Episodes', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('ablation_gnn.png')
plt.show()


plt.figure(figsize=(10, 5))
plt.plot(smooth_curve(res_full), label='With Cooperation (QMIX)', color='#1f77b4', linewidth=2.5)
plt.plot(smooth_curve(res_no_coop), label='Without Cooperation (Independent)', color='#d62728', linestyle='-.')

plt.title('Ablation Study: Effect of Multi-Agent Cooperation', fontsize=14)
plt.xlabel('Training Episodes', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('ablation_cooperation.png')
plt.show()

In [ ]:
def save_complete_model(gat, mod, fc, mixer, path="rumor_marl_model.pth"):
    
    checkpoint = {
        'gat_state_dict': gat.state_dict(),
        'mod_state_dict': mod.state_dict(),
        'fc_state_dict': fc.state_dict(),
    }
    
   
    if mixer is not None:
        checkpoint['mixer_state_dict'] = mixer.state_dict()
    
    torch.save(checkpoint, path)
    print(f"Successfully saved all models to {path}")


save_complete_model(gat_full, mod_full, fc_full, mixer_full,"Full_model.pth")
save_complete_model(gat_no_gnn,mod_no_gnn,fc_no_gnn,mixer_no_gnn,"No_gnn.pth")
save_complete_model(gat_no_coop,mod_no_coop,fc_no_coop,mixer_no_coop,"No_coop.pth")
